# 05 — Combined Cross-Tissue Report

Aggregate per-tissue subcluster reports (produced by `02_subcluster_cards.ipynb`) into a
single interactive HTML report and a combined PowerPoint deck.

**Workflow:**
1. For each tissue, locate the `{TISSUE}_subcluster_report/` directory from Notebook 02
2. Copy all `img/` and `de/` assets into a unified `OUTPUT_DIR/img/{tissue}/` layout
3. Reconstruct card metadata from saved CSV and image file lists
4. Render `combined_report.html.j2` → `OUTPUT_DIR/index.html`
5. Build a combined PPTX with tissue-title slides between subcluster slides

**Outputs (`OUTPUT_DIR/`):**
- `index.html` — interactive combined report (keep `img/` and `de/` alongside it)
- `img/{tissue}/{safe}/` — per-tissue/per-subcluster PNG files
- `de/{tissue}_{safe}.csv` — DE tables (downloadable)
- `combined_report.pptx` — combined PowerPoint deck

## 1. Imports

In [ ]:
import io
import json
import re
import shutil
from datetime import datetime
from pathlib import Path

import pandas as pd
from jinja2 import Environment, FileSystemLoader
from pptx import Presentation
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN
from pptx.util import Cm, Inches, Pt

print('Imports OK')

## 2. Configuration

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║                    CONFIGURATION — edit this cell                       ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# Map tissue name → path to the {TISSUE}_subcluster_report/ directory
# produced by Notebook 02 for that tissue.
TISSUE_REPORT_DIRS: dict[str, Path] = {
    "lung":  Path("results/disease_subclusters/lung/lung_subcluster_report"),
    "brain": Path("results/disease_subclusters/brain/brain_subcluster_report"),
    # add more tissues here …
}

# Where to write the combined report (img/ and de/ will be created inside)
OUTPUT_DIR: Path = Path("results/disease_subclusters/combined_report")

# ─────────────────────────────────────────────────────────────────────────────
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'img').mkdir(exist_ok=True)
(OUTPUT_DIR / 'de').mkdir(exist_ok=True)

print('Configuration loaded.')
for tissue, rdir in TISSUE_REPORT_DIRS.items():
    exists = '✓' if rdir.exists() else '✗ NOT FOUND'
    print(f'  {tissue}: {rdir}  {exists}')

## 3. Copy Assets & Reconstruct Card Metadata

In [ ]:
def _safe_label(label: str) -> str:
    safe = re.sub(r'[^\w\-]', '_', label)
    safe = re.sub(r'_+', '_', safe)
    return safe.strip('_')


def _copy_subcluster_assets(
    tissue: str,
    report_dir: Path,
    out_dir: Path,
) -> list[dict]:
    """
    Copy img/{safe}/ and de/ from a per-tissue report dir into
    out_dir/img/{tissue}/{safe}/ and out_dir/de/{tissue}_{safe}.csv.
    Returns a list of card metadata dicts (one per subcluster).
    """
    cards: list[dict] = []

    img_src = report_dir / 'img'
    de_src  = report_dir / 'de'

    if not img_src.exists():
        print(f'  [warn] No img/ dir in {report_dir}')
        return cards

    # Each subcluster has its own subdirectory under img/
    for safe_dir in sorted(img_src.iterdir()):
        if not safe_dir.is_dir():
            continue
        safe = safe_dir.name

        # Copy images
        dst_img = out_dir / 'img' / tissue / safe
        dst_img.mkdir(parents=True, exist_ok=True)
        for src_file in sorted(safe_dir.glob('*.png')):
            shutil.copy2(src_file, dst_img / src_file.name)

        # Build path refs (relative to OUTPUT_DIR/index.html)
        def _img(name):
            p = dst_img / name
            return f'img/{tissue}/{safe}/{name}' if p.exists() else None

        # Copy DE CSV
        de_csv_path = None
        de_json = None
        if de_src.exists():
            src_csv = de_src / f'{safe}.csv'
            if src_csv.exists():
                dst_csv = out_dir / 'de' / f'{tissue}_{safe}.csv'
                shutil.copy2(src_csv, dst_csv)
                de_csv_path = f'de/{tissue}_{safe}.csv'
                try:
                    df = pd.read_csv(dst_csv)
                    de_json = df.to_json(orient='split')
                except Exception:
                    de_json = None

        card = {
            'label':             safe.replace('_', ' '),  # approximation; overridden below
            'anchor':            safe,
            'disease':           safe,
            'stats':             {},
            'disease_breakdown': [],
            'umap_celltype_path':  _img('umap_celltype.png'),
            'umap_disease_path':   _img('umap_disease.png'),
            'umap_highlight_path': _img('umap_highlight.png'),
            'volcano_path':        _img('volcano.png'),
            'dotplot_path':        _img('enrichment.png'),
            'de_csv_path':         de_csv_path,
            'de_json':             de_json,
        }
        cards.append(card)

    # Try to enrich labels from subcluster_info CSV if present in parent dir
    info_csv = report_dir.parent / 'subcluster_info.csv'
    if not info_csv.exists():
        # Check for {tissue}_subcluster_info.csv pattern
        matches = list(report_dir.parent.glob('*_subcluster_info.csv'))
        if matches:
            info_csv = matches[0]

    if info_csv.exists():
        try:
            info_df = pd.read_csv(info_csv)
            safe_to_label = {_safe_label(row['label']): row['label']
                             for _, row in info_df.iterrows() if 'label' in info_df.columns}
            for card in cards:
                label = safe_to_label.get(card['anchor'], card['label'])
                card['label'] = label
                card['disease'] = label.split('|')[1] if '|' in label else label
                # Stats
                row = info_df[info_df['label'] == label]
                if not row.empty:
                    r = row.iloc[0]
                    card['stats'] = {
                        'n_total':    str(int(r['n_cells'])) if pd.notna(r.get('n_cells')) else '—',
                        'n_disease':  str(int(r['n_disease_cells'])) if pd.notna(r.get('n_disease_cells')) else '—',
                        'cell_type':  str(r.get('subset', '—')),
                        'fold_enrichment': f"{float(r['fold_enrichment']):.3g}" if pd.notna(r.get('fold_enrichment')) else '—',
                        'fdr_pval':   f"{float(r['pvalue_adj']):.2g}" if pd.notna(r.get('pvalue_adj')) else '—',
                    }
        except Exception as e:
            print(f'  [warn] Could not read subcluster_info from {info_csv}: {e}')

    return cards


# ── Main loop ────────────────────────────────────────────────────────────────────────
tissues: dict[str, list[dict]] = {}

for tissue, report_dir in TISSUE_REPORT_DIRS.items():
    if not report_dir.exists():
        print(f'⚠  {tissue}: report directory not found — skipping ({report_dir})')
        continue
    print(f'\n── {tissue}')
    cards = _copy_subcluster_assets(tissue, report_dir, OUTPUT_DIR)
    tissues[tissue] = cards
    print(f'  {len(cards)} subclusters copied')

total_subclusters = sum(len(v) for v in tissues.values())
print(f'\n✓ {len(tissues)} tissue(s), {total_subclusters} subclusters total.')
print(f'  Output dir: {OUTPUT_DIR.resolve()}')

## 4. Render Combined HTML Report

In [ ]:
if '__file__' in dir():
    TEMPLATE_DIR = Path(__file__).parent / 'templates'
elif (Path.cwd() / 'templates').is_dir():
    TEMPLATE_DIR = Path('templates')
else:
    TEMPLATE_DIR = Path('notebooks/disease_subclusters/templates')

env      = Environment(loader=FileSystemLoader(str(TEMPLATE_DIR)), autoescape=False)
template = env.get_template('combined_report.html.j2')

html_out = template.render(
    tissues=tissues,
    total_subclusters=total_subclusters,
    generated_at=datetime.now().strftime('%Y-%m-%d %H:%M'),
)

html_path = OUTPUT_DIR / 'index.html'
html_path.write_text(html_out, encoding='utf-8')
size_kb = html_path.stat().st_size / 1024
print(f'✓ Combined HTML written: {html_path}  ({size_kb:.0f} KB)')
print(f'  Open: {html_path.resolve()}')

## 5. Build Combined PowerPoint Deck

In [ ]:
SLIDE_W = Inches(13.33)
SLIDE_H = Inches(7.5)

_DARK  = RGBColor(0x1a, 0x1a, 0x2e)
_BLUE  = RGBColor(0x4a, 0x90, 0xd9)
_WHITE = RGBColor(0xff, 0xff, 0xff)
_LGREY = RGBColor(0xf0, 0xf2, 0xf5)
_TEXT  = RGBColor(0x1a, 0x1a, 0x2e)


def _add_text_box(slide, left, top, width, height, text, *,
                  bold=False, fontsize=10, color=_TEXT,
                  align=PP_ALIGN.LEFT, wrap=True):
    txBox = slide.shapes.add_textbox(left, top, width, height)
    tf = txBox.text_frame
    tf.word_wrap = wrap
    p = tf.paragraphs[0]
    p.text = text
    p.alignment = align
    run = p.runs[0] if p.runs else p.add_run()
    run.font.bold = bold
    run.font.size = Pt(fontsize)
    run.font.color.rgb = color
    return txBox


def _add_tissue_title_slide(prs, tissue: str, n_subclusters: int):
    """Insert a tissue-level divider slide."""
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    bg = slide.shapes.add_shape(1, 0, 0, SLIDE_W, SLIDE_H)
    bg.fill.solid()
    bg.fill.fore_color.rgb = _DARK
    bg.line.fill.background()
    _add_text_box(
        slide, Cm(4), Cm(2.5), Cm(25), Cm(3),
        tissue.upper(), bold=True, fontsize=36, color=_WHITE, align=PP_ALIGN.CENTER,
    )
    _add_text_box(
        slide, Cm(4), Cm(6), Cm(25), Cm(1.5),
        f'{n_subclusters} disease-enriched subcluster(s)',
        bold=False, fontsize=16, color=_BLUE, align=PP_ALIGN.CENTER,
    )


def _add_subcluster_slide(prs, tissue: str, card: dict):
    """Add a single subcluster slide with available images."""
    slide = prs.slides.add_slide(prs.slide_layouts[6])

    # Title bar
    bar = slide.shapes.add_shape(1, 0, 0, SLIDE_W, Cm(1.8))
    bar.fill.solid()
    bar.fill.fore_color.rgb = _DARK
    bar.line.fill.background()
    _add_text_box(slide, Cm(0.4), Cm(0.1), Cm(20), Cm(1.6),
                  card['label'], bold=True, fontsize=14, color=_WHITE)
    _add_text_box(slide, Cm(21), Cm(0.1), Cm(12), Cm(1.6),
                  f'Tissue: {tissue}', bold=False, fontsize=11, color=_BLUE,
                  align=PP_ALIGN.RIGHT)

    # Stats on left
    stats = card.get('stats', {})
    rows = [
        ('Total cells',     stats.get('n_total',         '—')),
        ('Disease cells',   stats.get('n_disease',       '—')),
        ('Cell type',       stats.get('cell_type',       '—')),
        ('Fold enrichment', stats.get('fold_enrichment', '—')),
        ('FDR',             stats.get('fdr_pval',        '—')),
    ]
    row_h = Cm(0.48)
    tbl = slide.shapes.add_table(
        len(rows), 2, Cm(0.4), Cm(2.0), Cm(12.7), row_h * len(rows)
    ).table
    tbl.columns[0].width = Cm(4.5)
    tbl.columns[1].width = Cm(8.2)
    for ri, (lbl, val) in enumerate(rows):
        for ci, (text, bold) in enumerate([(lbl, True), (str(val), False)]):
            cell = tbl.cell(ri, ci)
            if bold:
                cell.fill.solid(); cell.fill.fore_color.rgb = _LGREY
            tf = cell.text_frame
            tf.word_wrap = False
            p = tf.paragraphs[0]; p.clear()
            run = p.add_run()
            run.text = text
            run.font.bold = bold
            run.font.size = Pt(7)
            run.font.color.rgb = _TEXT

    # Images on right
    RIGHT_L = Cm(13.5)
    image_order = [
        ('umap_celltype_path',  'umap_celltype.png'),
        ('umap_disease_path',   'umap_disease.png'),
        ('umap_highlight_path', 'umap_highlight.png'),
        ('volcano_path',        'volcano.png'),
        ('dotplot_path',        'enrichment.png'),
    ]
    umap_imgs = [(k, n) for k, n in image_order[:3] if card.get(k)]
    other_imgs = [(k, n) for k, n in image_order[3:] if card.get(k)]

    if umap_imgs:
        umap_w = Cm(6.0)
        umap_top = Cm(2.0)
        for i, (key, _) in enumerate(umap_imgs):
            img_path = OUTPUT_DIR / card[key]
            if img_path.exists():
                try:
                    slide.shapes.add_picture(
                        str(img_path),
                        RIGHT_L + i * (umap_w + Cm(0.2)), umap_top,
                        width=umap_w,
                    )
                except Exception:
                    pass

    if other_imgs:
        plot_top = Cm(2.0) + Cm(4.0) if umap_imgs else Cm(2.0)
        plot_w = Cm(9.5)
        for i, (key, _) in enumerate(other_imgs):
            img_path = OUTPUT_DIR / card[key]
            if img_path.exists():
                try:
                    slide.shapes.add_picture(
                        str(img_path),
                        RIGHT_L + i * (plot_w + Cm(0.2)), plot_top,
                        width=plot_w,
                    )
                except Exception:
                    pass


# ── Build presentation ───────────────────────────────────────────────────────────────────
prs = Presentation()
prs.slide_width  = SLIDE_W
prs.slide_height = SLIDE_H

for tissue, cards in tissues.items():
    _add_tissue_title_slide(prs, tissue, len(cards))
    for card in cards:
        _add_subcluster_slide(prs, tissue, card)

pptx_path = OUTPUT_DIR / 'combined_report.pptx'
prs.save(str(pptx_path))
print(f'✓ Combined PPTX written: {pptx_path}  ({pptx_path.stat().st_size / 1024:.0f} KB)')

## Summary

| Output | Path |
|--------|------|
| Combined HTML report | `{OUTPUT_DIR}/index.html` |
| Per-tissue images | `{OUTPUT_DIR}/img/{tissue}/{safe}/` |
| DE tables (CSV) | `{OUTPUT_DIR}/de/{tissue}_{safe}.csv` |
| Combined PowerPoint | `{OUTPUT_DIR}/combined_report.pptx` |

**Viewing the report:** open `index.html` in a browser — the `img/` and `de/` folders must stay alongside it.  
**Sharing:** zip the entire `{OUTPUT_DIR}/` directory; all paths are relative.